# Week 2 – Preprocessing & Feature Engineering
## Fake News Detection Project

**Goal:** Clean the text data, engineer meaningful features, and prepare the final feature matrix for modeling.

In [1]:
# Imports
import pandas as pd
import numpy as np
import re
import string
import pickle
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
import scipy.sparse as sp
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded ✅')

Libraries loaded ✅


In [2]:
# Load raw data from Week 1
df = pd.read_csv('../data/raw_data.csv')
print(f'Loaded {len(df):,} rows')
df.head(3)

Loaded 44,898 rows


,title,text,subject,date,label,title_len,text_len
0,BREAKING: GOP Chairman Grassley Has Had Enoug...,"Donald Trump s White House is in chaos, and th...",News,"July 21, 2017",0,11,361
1,Failed GOP Candidates Remembered In Hilarious...,Now that Donald Trump is the presumptive GOP n...,News,"May 7, 2016",0,9,495
2,Mike Pence’s New DC Neighbors Are HILARIOUSLY...,Mike Pence is a huge homophobe. He supports ex...,News,"December 3, 2016",0,14,379


In [3]:
# ─────────────────────────────────────────
# STEP 1: Text Cleaning
# ─────────────────────────────────────────
def clean_text(text):
    """Lowercase, remove URLs, punctuation, extra spaces."""
    if pd.isna(text):
        return ''
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)   # URLs
    text = re.sub(r'<.*?>', '', text)                     # HTML tags
    text = re.sub(r'\d+', ' NUM ', text)                  # Numbers -> token
    text = text.translate(str.maketrans('', '', string.punctuation))  # Punctuation
    text = re.sub(r'\s+', ' ', text).strip()              # Whitespace
    return text

df['clean_title'] = df['title'].apply(clean_text)
df['clean_text']  = df['text'].apply(clean_text)
df['combined']    = df['clean_title'] + ' ' + df['clean_text']   # Merge title + body

print('Text cleaning done ✅')
print('Sample cleaned text:')
print(df['combined'].iloc[0][:200])

Text cleaning done ✅
Sample cleaned text:
breaking gop chairman grassley has had enough demands trump jr testimony donald trump s white house is in chaos and they are trying to cover it up their russia problems are mounting by the hour and th


In [4]:
# ─────────────────────────────────────────
# STEP 2: Handcrafted Features
# ─────────────────────────────────────────

# Exclamation / question marks (sensationalist style)
df['num_exclamations'] = df['text'].apply(lambda x: str(x).count('!'))
df['num_questions']    = df['text'].apply(lambda x: str(x).count('?'))

# Uppercase ratio (shouting = fake signal)
def uppercase_ratio(text):
    text = str(text)
    if len(text) == 0: return 0
    return sum(1 for c in text if c.isupper()) / len(text)

df['uppercase_ratio'] = df['text'].apply(uppercase_ratio)

# Average word length
def avg_word_len(text):
    words = str(text).split()
    if not words: return 0
    return np.mean([len(w) for w in words])

df['avg_word_len'] = df['clean_text'].apply(avg_word_len)

# Text and title lengths
df['text_word_count']  = df['clean_text'].apply(lambda x: len(x.split()))
df['title_word_count'] = df['clean_title'].apply(lambda x: len(x.split()))

# Unique word ratio (vocabulary richness)
def unique_word_ratio(text):
    words = str(text).split()
    if not words: return 0
    return len(set(words)) / len(words)

df['unique_word_ratio'] = df['clean_text'].apply(unique_word_ratio)

# Subject label encoding (if available)
if 'subject' in df.columns:
    le = LabelEncoder()
    df['subject_enc'] = le.fit_transform(df['subject'].fillna('unknown'))
    with open('../data/subject_encoder.pkl', 'wb') as f:
        pickle.dump(le, f)
    print('Subject encoder saved ✅')

print('\nHandcrafted features created ✅')
feature_cols = ['num_exclamations','num_questions','uppercase_ratio',
                'avg_word_len','text_word_count','title_word_count','unique_word_ratio']
if 'subject_enc' in df.columns:
    feature_cols.append('subject_enc')
print(f'Features: {feature_cols}')
df[feature_cols].describe().round(3)

Subject encoder saved ✅

Handcrafted features created ✅
Features: ['num_exclamations', 'num_questions', 'uppercase_ratio', 'avg_word_len', 'text_word_count', 'title_word_count', 'unique_word_ratio', 'subject_enc']


,num_exclamations,num_questions,uppercase_ratio,avg_word_len,text_word_count,title_word_count,unique_word_ratio,subject_enc
count,44898.000,44898.000,44898.000,44898.000,44898.000,44898.000,44898.000,44898.000
mean,0.408,0.679,0.044,4.848,406.408,12.480,0.572,4.720
std,1.456,1.795,0.035,0.700,352.583,4.141,0.133,2.052
min,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
25%,0.000,0.000,0.030,4.711,204.000,10.000,0.506,2.000
50%,0.000,0.000,0.038,4.928,363.000,11.000,0.558,5.000
75%,0.000,1.000,0.048,5.127,514.000,14.000,0.624,6.000
max,133.000,94.000,0.882,11.667,8156.000,44.000,1.000,7.000


In [5]:
# ─────────────────────────────────────────
# STEP 3: Train / Test Split (stratified)
# ─────────────────────────────────────────
X_text = df['combined']
X_meta = df[feature_cols].values
y      = df['label']

(
    X_text_train, X_text_test,
    X_meta_train, X_meta_test,
    y_train, y_test
) = train_test_split(X_text, X_meta, y, test_size=0.2,
                      random_state=42, stratify=y)

print(f'Train size : {len(y_train):,}  ({y_train.mean()*100:.1f}% real)')
print(f'Test  size : {len(y_test):,}   ({y_test.mean()*100:.1f}% real)')

Train size : 35,918  (47.7% real)
Test  size : 8,980   (47.7% real)


In [6]:
# ─────────────────────────────────────────
# STEP 4: TF-IDF Vectorization
# ─────────────────────────────────────────
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),       # unigrams + bigrams
    sublinear_tf=True,         # log scaling
    min_df=5,
    max_df=0.9,
    strip_accents='unicode'
)

X_tfidf_train = tfidf.fit_transform(X_text_train)
X_tfidf_test  = tfidf.transform(X_text_test)

print(f'TF-IDF shape train : {X_tfidf_train.shape}')
print(f'TF-IDF shape test  : {X_tfidf_test.shape}')

# Save vectorizer
with open('../data/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
print('TF-IDF vectorizer saved ✅')

TF-IDF shape train : (35918, 50000)
TF-IDF shape test  : (8980, 50000)
TF-IDF vectorizer saved ✅


In [7]:
# ─────────────────────────────────────────
# STEP 5: Combine TF-IDF + Handcrafted Features
# ─────────────────────────────────────────
from scipy.sparse import hstack, csr_matrix

X_train = hstack([X_tfidf_train, csr_matrix(X_meta_train)])
X_test  = hstack([X_tfidf_test,  csr_matrix(X_meta_test)])

print(f'Final feature matrix – Train : {X_train.shape}')
print(f'Final feature matrix – Test  : {X_test.shape}')

# Save for Week 3
sp.save_npz('../data/X_train.npz', X_train)
sp.save_npz('../data/X_test.npz',  X_test)
y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv',   index=False)

# Also save feature column names
import json
with open('../data/feature_cols.json', 'w') as f:
    json.dump(feature_cols, f)

print('\nAll preprocessed data saved ✅')
print('Ready for Week 3 – Modeling!')

Final feature matrix – Train : (35918, 50008)
Final feature matrix – Test  : (8980, 50008)

All preprocessed data saved ✅
Ready for Week 3 – Modeling!
